# ScamSentry ML evidence

**Deven Sachin Gaikwad** · B.Tech Electronics and Communication Engineering (AI and ML) · DES Pune University

Open in Colab with a CPU runtime and choose **Runtime → Run all**. The default path redraws the published evidence and writes charts and CSVs under **scamsentry_evidence/**. It needs only NumPy, pandas, Matplotlib and IPython. It fetches small, hash-checked public JSON files if the project checkout is absent.

ScamSentry uses character TF-IDF and calibrated multinomial logistic regression for historical English short messages. The three source classes are ordinary, promotional or spam, and smishing-like. Classifier scores describe text resemblance. Product warnings also use separate observations and abstention logic.

| Scope | Evidence in this notebook |
| --- | --- |
| Frozen training partition | Split counts, training lengths and lexical feature correlations |
| Frozen historical test | Saved support, confusion, precision, recall, F1, calibration and robustness |
| Known development partition | ROC and precision-recall curves from archived probabilities, plus earlier candidate and learning curves |
| V4 training-fold cross-validation | Candidate screening and the preregistered recall floor |
| Optional scratch training | A CPU demonstration of the baseline C2 configuration with separate development and calibration partitions |

Source corpus: Sandhya Mishra and Devpriya Soni (2022), [SMS Phishing Dataset for Machine Learning and Pattern Recognition, V1](https://doi.org/10.17632/f45bkkt8pr.1), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Published outputs here are aggregates. The project transformed identifiers, grouped templates, removed duplicates and used group-disjoint partitions. A shared notebook can expose saved cell outputs. Use the aggregate exports when sharing the evidence. [Dataset record](https://github.com/DevenGaikwad/ScamSentry/blob/main/docs/DATA_SOURCES.md) · [Model card](https://github.com/DevenGaikwad/ScamSentry/blob/main/docs/MODEL_CARD.md)

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import urllib.request

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import Image, display

RUN_TRAINING_DEMO = False
TRAINING_CSV = None  # Optional path to the publisher's exact Dataset_5971.csv
OUTPUT = Path(os.environ.get("SCAMSENTRY_OUTPUT_DIR", "scamsentry_evidence")).expanduser().resolve()
OUTPUT.mkdir(parents=True, exist_ok=True)

REPOSITORY = "DevenGaikwad/ScamSentry"
EVIDENCE_COMMIT = "01765677ea3e7a95b8983498168604cfec83090c"
ARCHIVED_MODEL_SHA256 = "14927b442dc9f33a35ad7086bd522b87e1110a73a6d1b2826b73ac1bf1a8a8fa"

INPUTS = {'data': ('01765677ea3e7a95b8983498168604cfec83090c',
          'ml/artifacts/data_summary.json',
          '157ced55c5b2cfb3c675c79a38017c201609b5d05bfd935e796e59a9a1b06516'),
 'splits': ('01765677ea3e7a95b8983498168604cfec83090c',
            'ml/artifacts/split_summary.json',
            '774fe36dda99ec8f2b70478e1461f977fc18596aad0f30980d513b3879123395'),
 'run': ('01765677ea3e7a95b8983498168604cfec83090c',
         'ml/artifacts/runs/baseline_char_lr_001/run.json',
         'a296455b02011ae6bfaffb37ede7290cd9f64a51bd375c242f8eb5b370d54996'),
 'final': ('01765677ea3e7a95b8983498168604cfec83090c',
           'ml/artifacts/runs/baseline_char_lr_001/final_test.json',
           '4175c9d4b19f30d157009b0f4592aff30f4bfb571baf478ad886ebdc0d501118'),
 'comparison': ('01765677ea3e7a95b8983498168604cfec83090c',
                'ml/artifacts/runs/baseline_char_lr_001/comparison.json',
                '1517bf9c2344e8095d7fdfe3c47bb37de48429af612316dce5f8a7cb611286e4'),
 'learning': ('01765677ea3e7a95b8983498168604cfec83090c',
              'ml/artifacts/research_comparison.json',
              '9ab1fa93e1338f305b375dec1f6df94ed6bcf9ec39a815048691f758685cf37b'),
 'robustness': ('01765677ea3e7a95b8983498168604cfec83090c',
                'ml/artifacts/robustness.json',
                '3c1db16c9601a630457018e110cbb9cb28cd75bbbe45b67d1cc5daaed4b1fe78'),
 'cv': ('01765677ea3e7a95b8983498168604cfec83090c',
        'ml/artifacts/v4/robust_hybrid_20261118_01/cv.json',
        '3d2416116a64c4f8c3bcf82221b7018b3a0b9f489c6ddb412d75619f8d646231'),
 'cv_summary': ('01765677ea3e7a95b8983498168604cfec83090c',
                'ml/artifacts/v4/robust_hybrid_20261118_01/summary.json',
                '4ac19995e8be326ec614668de7a73dface5794db0a5ee806cf2221c4fe6e15f9'),
 'cv_plan': ('01765677ea3e7a95b8983498168604cfec83090c',
             'ml/artifacts/v4/robust_hybrid_20261118_01/preregistration.json',
             '3ec9325be1869eba93a9ea74c106289f4750732f5e6b8d53dcdc74f4ec47d4fa'),
 'aggregates': ('main',
                'ml/notebooks/presentation_aggregates.json',
                '242b732427ed23423f377fb1ba644c2ccc15158b2457a78998cb9b9f08c0b8a2')}

In [ ]:
def load_evidence(name):
    ref, relative_path, expected = INPUTS[name]
    cwd = Path.cwd().resolve()
    local = next((parent / relative_path for parent in (cwd, *cwd.parents) if (parent / relative_path).is_file()), None)
    if local is not None:
        with local.open("rb") as evidence_file:
            content = evidence_file.read(2_000_001)
    else:
        url = f"https://raw.githubusercontent.com/{REPOSITORY}/{ref}/{relative_path}"
        request = urllib.request.Request(url, headers={"User-Agent": "ScamSentry ML evidence notebook"})
        with urllib.request.urlopen(request, timeout=30) as response:
            content = response.read(2_000_001)
    if len(content) > 2_000_000 or hashlib.sha256(content).hexdigest() != expected:
        raise ValueError(f"Evidence SHA-256 mismatch for {relative_path}. Open the matched release notebook.")
    return json.loads(content)


evidence = {key: load_evidence(key) for key in INPUTS}
dataset = evidence["data"]
splits = evidence["splits"]
archived_run = evidence["run"]
test = evidence["final"]
aggregate = evidence["aggregates"]
train = aggregate["training"]
rank = aggregate["development_ranking"]

assert test["protocol_id"] == aggregate["protocol_id"] == "ss-group-v1"
assert test["model_sha256"] == aggregate["model_sha256"] == ARCHIVED_MODEL_SHA256
assert test["split_sha256"] == aggregate["split_sha256"] == archived_run["split_digest"]
assert dataset["processed_sha256"] == archived_run["dataset_digest"] == aggregate["processed_sha256"]
assert archived_run["fit_partition"] == "train"
assert archived_run["selection_partition"] == "development"
assert archived_run["calibration_partition"] == "calibration"
assert train["partition"] == "train" and rank["partition"] == "development"
assert train["records"] == splits["train"]["records"] and train["groups"] == splits["train"]["groups"]
assert rank["records"] == splits["development"]["records"]
assert test["class_order"] == train["class_order"]

LABELS = ["Ordinary", "Promotion / spam", "Smishing-like"]
SHORT = ["Ordinary", "Promotion", "Smishing"]
CLASS_COLORS = ["#64748b", "#b56017", "#146a82"]
TEAL, ORANGE, INK = "#146a82", "#b56017", "#162235"
CHARTS = {}
mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "text.color": INK,
    "axes.labelcolor": INK, "axes.titlecolor": INK, "axes.edgecolor": "#cbd5e1",
    "xtick.color": INK, "ytick.color": INK, "font.size": 10, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 150,
})


def finish_chart(fig, filename, partition, records, groups=None):
    path = OUTPUT / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    CHARTS[filename] = {"partition": partition, "records": records}
    if groups is not None:
        CHARTS[filename]["groups"] = groups
    plt.close(fig)
    display(Image(filename=str(path)))


def percent_y(ax, upper=1.05):
    ax.set_ylim(0, upper)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis="y", alpha=.2)
    ax.set_axisbelow(True)


def heatmap(ax, values, xlabels, ylabels, color, low, high, render):
    image = ax.imshow(values, cmap=color, vmin=low, vmax=high, aspect="equal")
    ax.set_xticks(range(len(xlabels)), xlabels, rotation=24, ha="right")
    ax.set_yticks(range(len(ylabels)), ylabels)
    for i, row in enumerate(values):
        for j, value in enumerate(row):
            red, green, blue, _ = image.cmap(image.norm(value))
            luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
            ax.text(j, i, render(value), ha="center", va="center",
                    color="white" if luminance < .47 else INK, fontweight="semibold")
    return image


print(f"Verified {len(INPUTS)} source files. Model {archived_run['winner']} with temperature {archived_run['temperature']:.6f}.")
print(f"Historical test · {test['records']:,} messages across {splits['test']['groups']:,} groups. "
      f"Accuracy {test['accuracy']:.2%}, macro F1 {test['macro_f1']:.2%}.")

## Data preparation and the training split

The raw row count is 5,971. Conflicting-label groups account for 130 excluded records and 101 additional exact redundancies were removed. The 5,740 usable examples were assigned to four disjoint partitions. Training fits vocabulary and model weights, development compares candidates, calibration fits the temperature, and the archived test records the historical result.

Lengths and correlations use only the 3,444 minimized training records across 3,244 groups. Token markers represent the number and URL placeholders left by minimization. Distinct-token share is zero when a message has no tokens. Punctuation share is zero when a normalized message has no characters. Correlations describe co-movement among six numeric features. Length and token count may be related by construction. The heatmap is descriptive and does not measure feature importance or use class numbers as an ordered variable.

In [ ]:
partition_order = ["train", "development", "calibration", "test"]
split_table = pd.DataFrame([
    {"partition": partition, "records": splits[partition]["records"], "groups": splits[partition]["groups"],
     **{label: splits[partition]["classes"][str(i)] for i, label in enumerate(LABELS)}}
    for partition in partition_order
]).set_index("partition")
assert (split_table[LABELS].sum(axis=1) == split_table["records"]).all()
assert split_table["records"].sum() == dataset["usable_rows"]
assert dataset["input_rows"] - dataset["quarantined_records"] - dataset["deduplicated_rows"] == dataset["usable_rows"]
display(split_table)
split_table.to_csv(OUTPUT / "split_support.csv")

fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2), layout="constrained")
stage_names = ["Parsed", "After conflict\nquarantine", "After exact\ndeduplication"]
stage_values = [dataset["parsed_rows"], dataset["parsed_rows"] - dataset["quarantined_records"], dataset["usable_rows"]]
bars = axes[0].bar(stage_names, stage_values, color=["#a8bbc8", "#628ba0", TEAL])
axes[0].bar_label(bars, labels=[f"{n:,}" for n in stage_values], padding=4)
axes[0].set(ylim=(0, 6500), ylabel="Messages", title="Dataset preparation")
split_table[LABELS].plot.bar(stacked=True, color=CLASS_COLORS, rot=18, ax=axes[1])
axes[1].set(ylim=(0, 3850), xlabel="", ylabel="Messages", title="Support by partition and class")
axes[1].legend(loc="upper right", frameon=False)
finish_chart(fig, "01_dataset_and_splits.png", "corpus audit and frozen splits",
             dataset["usable_rows"], sum(row["groups"] for row in splits.values()))

length_counts = np.asarray(train["length_counts_by_class"], dtype=int)
edges = train["length_bin_edges"]
assert np.array_equal(length_counts.sum(axis=1), split_table.loc["train", LABELS].to_numpy())
length_bins = [f"{lo}–{hi - 1}" for lo, hi in zip(edges, edges[1:])]
length_bins[-1] = f"{edges[-2]}–{edges[-1]}"
fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
for label, count, color in zip(LABELS, length_counts, CLASS_COLORS):
    ax.plot(range(len(length_bins)), count / count.sum(), marker="o", lw=2, color=color,
            label=f"{label} (n={count.sum():,})")
ax.set_xticks(range(len(length_bins)), length_bins, rotation=20)
ax.set(xlabel="Normalized Unicode code points (length bins)", ylabel="Share within class",
       title="Message length in the frozen training split")
percent_y(ax, .66)
ax.legend(frameon=False)
finish_chart(fig, "02_training_length.png", "train", train["records"], train["groups"])

In [ ]:
names = train["feature_names"]
display(pd.DataFrame.from_dict(train["feature_definitions"], orient="index", columns=["Definition"]))
labels = ["Characters", "Tokens", "Distinct share", "Punctuation", "Number markers", "URL markers"]
for method in ("pearson", "spearman"):
    matrix = np.asarray(train[method])
    assert matrix.shape == (len(names), len(names)) and np.isfinite(matrix).all()
    assert np.allclose(matrix, matrix.T) and np.allclose(np.diag(matrix), 1)
    assert np.max(np.abs(matrix)) <= 1
    pd.DataFrame(matrix, index=names, columns=names).to_csv(OUTPUT / f"training_{method}.csv")
fig, ax = plt.subplots(figsize=(7.8, 6.2), layout="constrained")
image = heatmap(ax, np.asarray(train["spearman"]), labels, labels, "RdBu", -1, 1, lambda v: f"{v:+.2f}")
fig.colorbar(image, ax=ax, shrink=.75, label="Spearman rank correlation")
ax.set_title(f"Lexical feature correlation · frozen train, n={train['records']:,}", pad=14)
finish_chart(fig, "03_training_feature_correlation.png", "train", train["records"], train["groups"])

## Historical held-out performance

Rows of the confusion matrix are the source labels and columns are predicted labels. A row percentage is relative to that class's support. The diagonal is correct classification. The one-vs-rest counts keep false alarms and missed cases visible for each class. The macro F1 gives every class equal weight.

This archived report describes one historical source. The interval below includes the resampling unit, number of resamples and seed. The reported false-positive counts refer to classifier argmax. Live app warnings use a separate decision policy.

In [ ]:
matrix = np.asarray(test["confusion_matrix"], dtype=np.int64)
support = matrix.sum(axis=1)
predicted = matrix.sum(axis=0)
tp = np.diag(matrix)
fp, fn = predicted - tp, support - tp
tn = matrix.sum() - (tp + fp + fn)
precision, recall = tp / predicted, tp / support
f1 = 2 * tp / (2 * tp + fp + fn)
assert matrix.shape == (3, 3) and (matrix >= 0).all() and matrix.sum() == test["records"]
assert np.array_equal(support, split_table.loc["test", LABELS].to_numpy())
archived_classes = [test["class_report"][str(i)] for i in range(len(LABELS))]
assert np.array_equal(support, [row["support"] for row in archived_classes])
assert np.allclose(precision, [row["precision"] for row in archived_classes])
assert np.allclose(recall, [row["recall"] for row in archived_classes])
assert np.allclose(f1, [row["f1-score"] for row in archived_classes])
assert np.isclose(tp.sum() / matrix.sum(), test["accuracy"]) and np.isclose(f1.mean(), test["macro_f1"])
class_table = pd.DataFrame({
    "Support": support, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    "Precision": precision, "Recall": recall, "F1": f1,
}, index=LABELS)
display(class_table.round(4))
class_table.to_csv(OUTPUT / "historical_class_metrics.csv", index_label="class")
pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(OUTPUT / "historical_confusion_counts.csv", index_label="actual")
ratio = matrix / support[:, None]
pd.DataFrame(ratio, index=LABELS, columns=LABELS).to_csv(OUTPUT / "historical_confusion_row_rates.csv", index_label="actual")
low, high = test["group_bootstrap_macro_f1_95ci"]
bootstrap = test["bootstrap"]
print(f"Historical macro F1 95% interval {low:.4f}–{high:.4f}. "
      f"Method: {bootstrap['method']} bootstrap, unit: {bootstrap['unit']}, "
      f"{bootstrap['repetitions']} resamples, seed {bootstrap['seed']}.")
display(pd.DataFrame([{
    "Test records": test["records"], "Test groups": splits["test"]["groups"],
    "Accuracy": test["accuracy"], "Macro F1": test["macro_f1"],
    "Group-bootstrap lower": low, "Group-bootstrap upper": high,
    "Smishing AP": test["smishing_average_precision"], "Smishing ROC-AUC": test["smishing_roc_auc"],
    "Log loss": test["log_loss"], "Multiclass Brier (summed)": test["brier_multiclass_sum"],
    "ECE (10 bins)": test["ece_10_equal_width"],
}]).round(5))

fig, axes = plt.subplots(1, 2, figsize=(10.3, 4.6), layout="constrained")
heatmap(axes[0], matrix, SHORT, SHORT, "Blues", 0, matrix.max(), lambda v: f"{v:d}")
heatmap(axes[1], ratio, SHORT, SHORT, "Blues", 0, 1, lambda v: f"{v:.1%}")
for ax, subtitle in zip(axes, ["Counts", "% within true class"]):
    ax.set(xlabel="Predicted label", ylabel="True label", title=subtitle)
fig.suptitle(f"Historical test confusion · n={matrix.sum():,}", fontweight="bold")
finish_chart(fig, "04_historical_confusion.png", "historical test", test["records"], splits["test"]["groups"])

fig, axes = plt.subplots(1, 2, figsize=(10.7, 4.0), layout="constrained")
class_table[["Precision", "Recall", "F1"]].plot.bar(color=["#6b8394", TEAL, "#c78c4d"], ax=axes[0], rot=18)
percent_y(axes[0])
axes[0].legend(frameon=False, loc="lower left")
axes[0].set(title="Precision, recall and F1")
class_table[["FP", "FN"]].plot.bar(color=["#8e5a39", "#983c48"], ax=axes[1], rot=18)
for bars in axes[1].containers:
    axes[1].bar_label(bars, padding=2)
axes[1].set(ylim=(0, max(fp.max(), fn.max()) + 4), ylabel="Errors", title="One-vs-rest errors by class")
axes[1].legend(frameon=False)
fig.suptitle("Historical test · class performance", fontweight="bold")
finish_chart(fig, "05_historical_class_performance.png", "historical test", test["records"], splits["test"]["groups"])

## Ranking, confidence and robustness

The ROC and precision-recall plots use the archived v2 baseline **development** scores. The plot markers come from a fixed score-threshold grid at 0.01 intervals. Exact AUROC and average precision use the complete saved score set. The horizontal PR reference is the positive-class share. The historical test section shows its saved scalar AP and AUROC separately.

The reliability diagram uses archived **historical test** top-class confidence bins. Circle size and the lower panel convey support. Calibration error depends on binning and dataset. The smallest bins have high sampling noise.

The robustness chart uses the four reported perturbation slices of the historical test. The OCR-like severe substitution is a text stress test. It measures sensitivity to synthetic transcription errors.

In [ ]:
rec = np.asarray(rank["recall"])
fpr = np.asarray(rank["false_positive_rate"])
prec = np.asarray(rank["precision"])
assert rec.shape == fpr.shape == prec.shape == (101,)
assert np.all(np.diff(rec) <= 1e-12) and np.all(np.diff(fpr) <= 1e-12)
assert rec[0] == fpr[0] == 1 and rec[-1] == fpr[-1] == 0
prevalence = rank["positive_support"] / rank["records"]
assert rank["positive_support"] == split_table.loc["development", "Smishing-like"]
fig, axes = plt.subplots(1, 2, figsize=(10.3, 4.2), layout="constrained")
axes[0].plot(fpr[::-1], rec[::-1], color=TEAL, lw=2.4, label=f"Exact AUROC {rank['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="#8293a3", label="Random ranking")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
axes[1].step(rec[::-1], prec[::-1], where="post", color=TEAL, lw=2.3,
             label=f"Exact average precision {rank['average_precision']:.3f}")
axes[1].axhline(prevalence, ls="--", color=ORANGE, label=f"Class share {prevalence:.1%}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–recall")
for ax in axes:
    ax.set(xlim=(0, 1.01), ylim=(0, 1.04))
    ax.grid(alpha=.16)
    ax.legend(loc="lower right", frameon=False)
fig.suptitle(f"Smishing-like vs rest · archived development, n={rank['records']}", fontweight="bold")
finish_chart(fig, "06_development_ranking.png", "development", rank["records"], splits["development"]["groups"])

In [ ]:
bins = pd.DataFrame(test["reliability"])
assert bins["n"].sum() == test["records"]
ece = np.average(np.abs(bins["accuracy"] - bins["confidence"]), weights=bins["n"])
assert np.isclose(ece, test["ece_10_equal_width"])
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0), layout="constrained")
axes[0].plot([.25, 1], [.25, 1], "--", color="#8293a3", label="Ideal calibration")
axes[0].scatter(bins["confidence"], bins["accuracy"], s=30 + 30*np.log10(bins["n"]), color=TEAL, zorder=3)
axes[0].set(xlim=(.25, 1.02), ylim=(-.06, 1.04), xlabel="Mean confidence", ylabel="Observed top-class accuracy",
            title=f"Reliability · ECE {ece:.3%}")
axes[0].legend(loc="upper left", frameon=False)
axes[1].bar(np.arange(len(bins)), bins["n"], color=TEAL, log=True)
axes[1].set_xticks(np.arange(len(bins)), [f"{p:.1f}–{p+.1:.1f}" for p in bins["lower"]], rotation=25)
axes[1].set(xlabel="Top-class confidence bin", ylabel="Records (log scale)", title="Support in nonempty bins")
fig.suptitle("Historical test · archived calibration", fontweight="bold")
finish_chart(fig, "07_historical_calibration.png", "historical test", test["records"], splits["test"]["groups"])

slices = evidence["robustness"]["families"]
slice_order = ["case", "spacing", "zero_width", "ocr_substitution"]
slice_labels = ["Clean", "Case", "Whitespace", "Zero-width", "Severe substitution"]
assert all(slices[name]["records"] == test["records"] for name in slice_order)
slice_f1 = [test["macro_f1"], *[slices[name]["macro_f1"] for name in slice_order]]
fig, ax = plt.subplots(figsize=(9.4, 4.1), layout="constrained")
bars = ax.bar(slice_labels, slice_f1, color=[TEAL, TEAL, TEAL, "#94652a", "#983c48"])
ax.bar_label(bars, labels=[f"{v:.1%}" for v in slice_f1], padding=4)
percent_y(ax)
ax.set(ylabel="Macro F1", title=f"Historical test · text perturbations, n={test['records']} per condition")
finish_chart(fig, "08_historical_robustness.png", "historical test", test["records"], splits["test"]["groups"])

## Training comparisons and the V4 decision

The single-seed learning curve and earlier estimator comparison evaluate the known development split. V4 evaluates candidates on three grouped folds inside the training partition. The frozen historical test was not rescored for these decisions.

The V4 recall floor was set at 82% before the run. The best clean cross-validation recall was approximately 81.9562%. None of the candidates passed selection, so the approved character-LR bundle remains in production. Do not compare a V4 CV value with the archived test as if both came from the same examples.

In [ ]:
curve = pd.DataFrame(evidence["learning"]["learning_curve"])
early = pd.DataFrame(evidence["comparison"]).sort_values("development_macro_f1")
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), layout="constrained")
axes[0].plot(curve["training_groups"], curve["development_macro_f1"], marker="o", color=TEAL, lw=2.5)
for k, v in zip(curve["training_groups"], curve["development_macro_f1"]):
    axes[0].annotate(f"{v:.1%}", (k, v), xytext=(0, 8), textcoords="offset points", ha="center")
axes[0].set(xlabel="Training groups used", ylabel="Development macro F1", ylim=(.75, .96),
            title="Learning curve · three single-seed samples")
axes[0].yaxis.set_major_formatter(PercentFormatter(1))
bars = axes[1].barh(early["candidate"], early["development_macro_f1"],
                    color=[TEAL if name == archived_run["winner"] else "#91a6b3" for name in early["candidate"]])
axes[1].bar_label(bars, labels=[f"{v:.1%}" for v in early["development_macro_f1"]], padding=4)
axes[1].set(xlim=(0, 1.07), xlabel="Development macro F1", title="Earlier estimator comparison")
axes[1].xaxis.set_major_formatter(PercentFormatter(1))
finish_chart(fig, "09_training_comparisons.png", "development", splits["development"]["records"],
             splits["development"]["groups"])

cv_summary, cv_plan = evidence["cv_summary"], evidence["cv_plan"]
cv = pd.DataFrame([{"candidate": item["candidate"], "eligible": item["eligible"], **item["mean"]}
                   for item in evidence["cv"]])
gate = cv_plan["config"]["cv_min_smishing_recall"]
assert cv_summary["fits"] == len(cv) * cv_plan["config"]["folds"]
assert not cv_summary["historical_test_scored"] and not cv["eligible"].any()
assert cv["smishing_recall"].max() < gate and cv_summary["promotion"] == "RETAIN_SHIPPING_BASELINE"
cv.to_csv(OUTPUT / "v4_training_fold_summary.csv", index=False)
names = ["Character", "Word", "Equal hybrid", "Char-heavy hybrid"]
pos = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(10.9, 4.2), layout="constrained")
axes[0].barh(pos - .18, cv["clean_macro_f1"], height=.32, color=TEAL, label="Clean")
axes[0].barh(pos + .18, cv["mild_macro_f1"], height=.32, color="#95aeb9", label="Perturbed")
axes[0].set(xlabel="Mean macro F1", xlim=(0, 1), title="Grouped validation folds")
axes[0].legend(loc="lower center", bbox_to_anchor=(.5, 1.06), ncols=2, frameon=False)
axes[1].scatter(cv["smishing_recall"], pos, color=TEAL, s=58)
for observed, y in zip(cv["smishing_recall"], pos):
    axes[1].annotate(f"{observed:.3%}", (observed, y), xytext=(-9, 0),
                     textcoords="offset points", ha="right", va="center", fontsize=9)
axes[1].axvline(gate, ls="--", color=ORANGE, label="82% floor")
axes[1].set(xlim=(.791, .826), xlabel="Mean clean smishing-like recall", title="Preregistered recall threshold")
axes[1].legend(frameon=False, loc="lower center", bbox_to_anchor=(.5, 1.06))
for ax in axes:
    ax.set_yticks(pos, names)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.grid(axis="x", alpha=.18)
    ax.set_axisbelow(True)
axes[1].set_xticks(np.arange(.795, .826, .005))
axes[1].xaxis.set_major_formatter(PercentFormatter(1, decimals=1))
fig.suptitle(f"V4 candidate screen · {cv_summary['fits']} fits, three folds per candidate", fontweight="bold")
finish_chart(fig, "10_v4_candidate_screen.png", "grouped cross-validation within train",
             train["records"], train["groups"])

## Optional exact-source training demonstration

Enable **RUN_TRAINING_DEMO** in the configuration cell and run all in a Python 3.12 CPU runtime to fit the baseline C2 configuration again. This creates a temporary checkout of the evidence source and an isolated virtual environment, installs the repository lockfile and downloads the registered CC BY 4.0 corpus. The existing fetch script checks archive and CSV SHA-256 digests. If the publisher download is unavailable, set **TRAINING_CSV** to a runtime copy of the exact registered CSV. The notebook verifies that file's SHA-256 before using it. Preparation and split verification run before the fit.

The cell displays and exports an aggregate verification receipt with the run ID, input digests, development metrics, and calibration scope. The run ID is separate and final-test scoring and app export are outside this path. A numerical replay can vary with the runtime. The published historical results and model remain the reference for the shipped app. All scratch training inputs and models are removed at the end of this cell.

In [ ]:
if RUN_TRAINING_DEMO:
    import subprocess
    import sys
    import tempfile
    import venv

    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("Use Python 3.12 for the locked optional training demo.")

    def run_command(stage, args, cwd, env=None):
        try:
            completed = subprocess.run(args, cwd=cwd, env=env, timeout=300, capture_output=True)
        except subprocess.TimeoutExpired:
            raise RuntimeError(f"{stage} timed out after 300 seconds.") from None
        if completed.returncode:
            # Keep full logs and row-level data out of shared outputs.
            raise RuntimeError(f"{stage} failed (exit {completed.returncode}).")

    with tempfile.TemporaryDirectory(prefix="scamsentry-colab-") as scratch:
        workspace = Path(scratch)
        checkout = workspace / "source"
        checkout.mkdir()
        run_command("Initialize temporary checkout", ["git", "init", "--quiet"], checkout)
        run_command("Set public source", ["git", "remote", "add", "origin", f"https://github.com/{REPOSITORY}.git"], checkout)
        run_command("Fetch frozen source", ["git", "fetch", "--quiet", "--depth", "1", "origin", EVIDENCE_COMMIT], checkout)
        run_command("Select frozen source", ["git", "checkout", "--quiet", "--detach", EVIDENCE_COMMIT], checkout)
        actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
        if actual != EVIDENCE_COMMIT:
            raise RuntimeError("Training source identity differs from the frozen evidence source.")
        ml = checkout / "ml"
        venv.EnvBuilder(with_pip=True).create(workspace / "venv")
        py = str(workspace / "venv" / "bin" / "python")
        env = {**os.environ, "PYTHONPATH": str(ml / "src"), "OPENBLAS_NUM_THREADS": "2",
               "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}
        run_command("Install locked dependencies", [py, "-m", "pip", "install", "--disable-pip-version-check",
                                                      "-r", "requirements-lock.txt"], ml, env)
        registry = json.loads((ml / "manifests/sources.json").read_text())
        registered = next(item for item in registry if item["source_id"] == "mendeley_smishing_v1")
        destination = ml / "data/raw/Dataset_5971.csv"
        if TRAINING_CSV:
            with Path(TRAINING_CSV).expanduser().open("rb") as provided:
                content = provided.read(2_000_001)
            if len(content) > 2_000_000 or hashlib.sha256(content).hexdigest() != registered["download_sha256"]:
                raise ValueError("The supplied CSV does not match the SHA-256 of the registered publisher file.")
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
        else:
            run_command("Download registered dataset or provide its verified CSV in TRAINING_CSV",
                        [py, str(checkout / "scripts/fetch_source.py")], checkout, env)
        csv_sha = hashlib.sha256(destination.read_bytes()).hexdigest()
        assert csv_sha == registered["download_sha256"]
        run_command("Prepare dataset", [py, "-m", "scamsentry_ml.data", "prepare",
                                        "--config", "configs/data.yaml"], ml, env)
        run_command("Verify frozen split", [py, "-m", "scamsentry_ml.splits", "verify",
                                            "--manifest", "artifacts/splits.json"], ml, env)
        config = {
            **archived_run["config"],
            "run_id": "colab_demo_char_lr_c2",
            "protocol_id": "ss-group-v1-colab-demonstration",
            "candidate_C": [2.0],
        }
        config_path = workspace / "train.json"
        config_path.write_text(json.dumps(config), encoding="utf-8")
        run_command("Fit and calibrate C2 demonstration",
                    [py, "-m", "scamsentry_ml.train", "--config", str(config_path)], ml, env)
        result = ml / "artifacts/runs/colab_demo_char_lr_c2"
        fit = json.loads((result / "run.json").read_text())
        development = json.loads((result / "development.json").read_text())
        assert fit["run_id"] == config["run_id"]
        assert fit["winner"] == "char_lr_C2" and not fit["final_test_accessed"]
        assert fit["dataset_digest"] == aggregate["processed_sha256"]
        assert fit["split_digest"] == aggregate["split_sha256"]
        assert development["records"] == splits["development"]["records"]
        assert (fit["fit_partition"], fit["selection_partition"], fit["calibration_partition"]) == (
            "train", "development", "calibration"
        )
        numpy_version = subprocess.check_output(
            [py, "-c", "from importlib.metadata import version; print(version('numpy'))"],
            cwd=ml, env=env, text=True, timeout=30
        ).strip()
        demo = {
            "schema_version": 1,
            "run_id": fit["run_id"],
            "estimator": fit["winner"],
            "training_source_commit": actual,
            "source_csv_sha256": csv_sha,
            "source_route": "verified supplied CSV" if TRAINING_CSV else "registered publisher archive",
            "processed_dataset_sha256": fit["dataset_digest"],
            "split_sha256": fit["split_digest"],
            "fit_records": splits["train"]["records"],
            "development_records": development["records"],
            "development_accuracy": development["accuracy"],
            "development_macro_f1": development["macro_f1"],
            "calibration_partition": fit["calibration_partition"],
            "calibration_records": splits["calibration"]["records"],
            "calibration_temperature": fit["temperature"],
            "test_scored": fit["final_test_accessed"],
            "runtime": {"python": fit["environment"]["python"], "numpy": numpy_version,
                        "scikit_learn": fit["environment"]["sklearn"], "matplotlib": mpl.__version__},
        }
        (OUTPUT / "optional_training_demo.json").write_text(json.dumps(demo, indent=2) + "\n", encoding="utf-8")
        display(pd.DataFrame([{
            "Run ID": demo["run_id"], "Development records": demo["development_records"],
            "Development accuracy": demo["development_accuracy"], "Development macro F1": demo["development_macro_f1"],
            "Calibration records": demo["calibration_records"], "Calibration temperature": demo["calibration_temperature"],
            "Test scored": demo["test_scored"],
        }]).round(6))
        print("Source and split hashes verified. Aggregate receipt saved to optional_training_demo.json.")
else:
    print("Saved-evidence mode. The optional scratch fit is disabled.")

## Export and interpretation

Use the PNGs in a report, presentation or README when the captions identify the partition and source. **scamsentry_evidence/** also contains the split table, model confusion and class metrics, both feature-correlation matrices, the V4 fold summary and the figure manifest.

The frozen historical benchmark and the development views come from one source. Real-world campaign coverage, prospective validation, language generalization and device performance require separately designed evaluation. Synthetic perturbations reveal test sensitivity. Shared notebook outputs and the README use aggregate evidence only. [Historical report](https://github.com/DevenGaikwad/ScamSentry/blob/main/docs/EVALUATION_RESULTS.md) · [V4 result](https://github.com/DevenGaikwad/ScamSentry/blob/main/docs/ML_V4_RESULTS.md)

In [ ]:
report = {
    "schema_version": 2,
    "evidence_commit": EVIDENCE_COMMIT,
    "model_sha256": ARCHIVED_MODEL_SHA256,
    "protocol_id": test["protocol_id"],
    "default_mode": "saved public aggregates",
    "figure_count": len(CHARTS),
    "figures": {name: hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest() for name in CHARTS},
    "figure_partitions": CHARTS,
    "sources": {key: {"ref": ref, "path": path, "sha256": digest} for key, (ref, path, digest) in INPUTS.items()},
    "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                "pandas": pd.__version__, "matplotlib": mpl.__version__},
}
assert len(CHARTS) == 10
(OUTPUT / "figure_manifest.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Exported {len(CHARTS)} figures, seven CSVs and a SHA-256 manifest with figure partitions to {OUTPUT.name}/.")